# 01 · Build the clean wildfire dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NamanArora332/ENGR_418_Team_3_Wildfire_Prediction/blob/main/notebooks/01_build_dataset.ipynb)

**Owner:** Role A (data lead) · **Runtime:** about 10–15 minutes on Colab, mostly downloading

This notebook turns the raw FPA FOD-Attributes CSVs into **one clean table** that the whole team trains on.

| Step | What happens |
|---|---|
| 1. Download | Get one CSV per year from Zenodo (about 150–250 MB each). Skipped if already downloaded. |
| 2. Inspect | List every column and put it into a leakage group (static, weather, outcome, …) |
| 3. Build | Filter states, remove duplicates and bad rows, fix types, add the target and calendar features |
| 4. Check | Class balance per year, and which columns each feature tier allows |

**Outputs** (in the shared Google Drive folder `ENGR418_wildfire_data/processed/`):
- `fires_clean.parquet`: the table everyone trains on
- `column_policy.csv`: every column, its group, missing %, and why
- `build_report.json`: how many rows each cleaning step removed

Everything heavy lives in `src/wildfire/`. This notebook just runs it and shows the results.

In [ ]:
# --- Setup: works on Google Colab and on your own computer ---------------------
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/NamanArora332/ENGR_418_Team_3_Wildfire_Prediction.git"
BRANCH = "main"   # change this if you want to test code from another branch
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/ENGR_418_Team_3_Wildfire_Prediction")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
else:
    repo = Path.cwd()
    while not (repo / "src" / "wildfire").exists() and repo != repo.parent:
        repo = repo.parent

sys.path.insert(0, str(repo / "src"))
print("Using code from:", repo)

## Settings
Change these, then run all cells (**Runtime → Run all**).

- `YEARS`: fire years to include. Ten years gives a fair split by year (train 2011–16, validate 2017–18, test 2019–20).
- `STATES`: western US by default. Set to `None` for all of the contiguous US (about 3× bigger).
- `DRY_RUN = True` uses small **fake** data, so you can test the code in seconds without downloading anything.

In [ ]:
from wildfire.config import BuildConfig, DEFAULT_YEARS, WESTERN_STATES, default_data_root

YEARS = DEFAULT_YEARS            # (2011, ..., 2020)
STATES = WESTERN_STATES          # ("AZ", "CA", "CO", "ID", "MT", "NM", "NV", "OR", "UT", "WA", "WY")
DRY_RUN = os.environ.get("WILDFIRE_DRY_RUN") == "1"   # set True to test with fake data

if DRY_RUN:
    DATA_ROOT = Path(os.environ.get("WILDFIRE_DATA_DIR", "/tmp/wildfire_dry_run"))
else:
    DATA_ROOT = default_data_root()   # Colab: MyDrive/ENGR418_wildfire_data, locally: ./data

# Raw CSVs are big (~2 GB): on Colab keep them on the fast local disk, not in Drive.
RAW_DIR = Path("/content/raw") if (IN_COLAB and not DRY_RUN) else DATA_ROOT / "raw"

cfg = BuildConfig(years=tuple(YEARS), states=STATES, data_root=DATA_ROOT, raw_dir=RAW_DIR)
print("Data folder:", cfg.data_root)
print("Raw CSVs:   ", cfg.raw_dir)
print("Years:", cfg.years[0], "to", cfg.years[-1], "| States:", cfg.states or "all")

## 1 · Download the raw CSVs

In [ ]:
if DRY_RUN:
    from wildfire.synthetic import write_synthetic_years
    print("DRY RUN: writing FAKE data. Do not report any numbers from it.")
    write_synthetic_years(cfg.raw_dir, cfg.years, n=3000)
else:
    from wildfire.download import download_years
    download_years(cfg.years, cfg.raw_dir)

## 2 · Inspect the columns
Every column goes into one group. `review` means no rule matched: **look at these and add a rule in `src/wildfire/columns.py`**.
Until then they're kept out of the features, which is the safe default.

In [ ]:
import pandas as pd
from wildfire.columns import classify_columns

pd.set_option("display.max_colwidth", 120, "display.max_rows", 400)

header = pd.read_csv(cfg.raw_path(cfg.years[-1]), nrows=5)
raw_policy = classify_columns(header.columns)
print(f"{len(header.columns)} columns in {cfg.raw_path(cfg.years[-1]).name}\n")
print(raw_policy["group"].value_counts().to_string())

In [ ]:
# Columns the model must never see, and why
raw_policy[raw_policy["group"].isin(["drop", "outcome", "retrospective"])].sort_values("group")

In [ ]:
# Columns no rule recognises yet: decide for each one, then add a rule in columns.py
raw_policy[raw_policy["group"] == "review"]

## 3 · Build the clean table

In [ ]:
from wildfire.clean import build_dataset

df, policy, report = build_dataset(cfg)
if DRY_RUN:
    (cfg.data_root / "processed" / "SYNTHETIC_DATA.txt").write_text("Built from fake data by DRY_RUN.")

print(f"\nClean table: {len(df):,} fires x {df.shape[1]} columns")
print("Saved to:", cfg.clean_path)

In [ ]:
# What each cleaning step removed
steps = ["rows_after_state_filter", "removed_duplicate_fod_id", "removed_missing_or_negative_size",
         "removed_bad_location", "removed_bad_date", "rows_final"]
pd.Series({s: report[s] for s in steps}, name="rows").to_frame()

In [ ]:
print("Sentinel codes (-9999 etc.) turned into missing values:", report["sentinel_values_set_to_nan"])
print("Text columns converted to numbers:", report["text_columns_converted_to_numbers"])
print("Dropped empty columns:", report["dropped_empty_columns"])
print("Dropped constant columns:", report["dropped_constant_columns"])

## 4 · Check the result

In [ ]:
# Class balance per year: Large fires are rare, so plain accuracy is misleading
counts = pd.DataFrame(report["class_counts_by_year"]).T.fillna(0).astype(int)
counts["Large %"] = (100 * counts["Large"] / counts.sum(axis=1)).round(2)
counts

In [ ]:
from wildfire.columns import FEATURE_TIERS, feature_columns

for tier, groups in FEATURE_TIERS.items():
    print(f"{tier:<14} {len(feature_columns(policy, tier)):>4} features  (groups: {', '.join(groups)})")

print("\nStill needing review:", report["columns_needing_review"] or "none")

In [ ]:
# The full column policy (also saved as column_policy.csv)
policy

## Done: what to tell the team
- The clean table is at `ENGR418_wildfire_data/processed/fires_clean.parquet` in the shared Drive folder.
- Teammates load it in their own notebooks with:

```python
from wildfire import load_clean, feature_columns, year_split
df, policy = load_clean()
X_cols = feature_columns(policy, tier="at_discovery")   # leakage-safe default
train, val, test = year_split(df)
```

Next: `02_eda.ipynb` for the charts in the report's data section.